# Transformer for Multimodal Stock Forecasting
Implementation notebook for *Adaptive Portfolio Management in Volatile Markets with Multiplex Attention Transformers and Deep Reinforcement Learning*.

Code for **PyTorch** implementation of the Transformer proposed in our paper, trained to predict multi-step stock price trajectories from:
1. **Technical indicators** (OHLCV-derived)
2. **News‑sentiment embeddings** (from FinBERT)
3. **Macroeconomic features**

The model outputs \(H\)-step forecasts that later serve as inputs to the downstream RL trading agent.

In [ ]:
!pip install -U torch torchvision torchaudio --quiet
!pip install -U transformers datasets peft --quiet

In [ ]:
import math, os, random, json, itertools, warnings, datetime as dt, typing as typ
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Running on', device)

In [ ]:
# some hyperparameters
WINDOW_SIZE = 30  # T
HORIZON_SIZE = 5  # H
BATCH_SIZE = 64

# feature dimensions
TECH_DIM = 10
SENT_DIM = 1
MACRO_DIM = 3

## Data Processing

In [ ]:
import pandas as pd, numpy as np

# CONFIG
TECH_CSV  = "train_data.csv"      #  must contain: date, ticker, tech columns
SENT_CSV  = "all_tickers_daily_sentiment.csv"      #  must contain: date, ticker, sentiment cols
MACRO_CSV = "DJIA_macro_combined.csv"     #  must contain: date, ticker, macro cols
LABEL_CSV = "train_data.csv"  # contains: date, ticker, open, close, high, low

TECH_COLS  = ['volume','macd','boll_ub','boll_lb','rsi_30','cci_30',
              'dx_30','close_30_sma','close_60_sma','vix']
SENT_COLS  = ['sentiment']
MACRO_COLS = ['fedfunds','sp500','sector']
LABEL_COLS = ['open','close','high','low']



import pandas as pd, numpy as np, torch
from torch.utils.data import Dataset, DataLoader


def load_tensor(csv_path: str, feature_cols: list[str],
                tickers=None, dates=None):
    df = pd.read_csv(csv_path, parse_dates=['date'])

    if tickers is None: tickers = sorted(df['ticker'].unique())
    if dates   is None: dates   = sorted(df['date'].unique())

    df = (df.query('ticker in @tickers and date in @dates')
            .sort_values(['date', 'ticker']))

    mats = []
    for feat in feature_cols:
        mat = (df.pivot(index='date', columns='ticker', values=feat)
                 .reindex(index=dates, columns=tickers)
                 .astype('float32')
                 .fillna(0.0)   # replace missing with 0, try avoiding nan
                 .values)
        mats.append(mat[..., None])

    return np.concatenate(mats, axis=2), tickers, dates



# build aligned tensors
tech, tickers, dates  = load_tensor(TECH_CSV,  TECH_COLS)
sent, _, _ = load_tensor(SENT_CSV,  SENT_COLS,  tickers, dates)
macro, _, _  = load_tensor(MACRO_CSV, MACRO_COLS, tickers, dates)

#  build 4‑price cube (Date × Ticker × 4)
lbl_df = pd.read_csv(LABEL_CSV, parse_dates=['date'])
lbl_df = (
    lbl_df.query('ticker in @tickers and date in @dates')
          .sort_values(['date', 'ticker'])
)

label_mats = []
for col in LABEL_COLS:         # ['open','close','high','low']
    mat = (lbl_df.pivot(index='date', columns='ticker', values=col)
                  .reindex(index=dates, columns=tickers)     # enforce exact order
                  .astype('float32')
                  .values)                                   # (D, N)
    label_mats.append(mat[..., None])                        # add 1‑len feature axis

labels = np.concatenate(label_mats, axis=2)         # (D, N, 4)



# feature scaling (min‑max to 0‑1)
def minmax_scale(arr, eps=1e-8):
    mn = np.nanmin(arr, axis=(0,1), keepdims=True)
    mx = np.nanmax(arr, axis=(0,1), keepdims=True)
    return (arr - mn) / (mx - mn + eps), mn, mx   # eps avoids /0 → NaN

tech_scaled,  tech_min,  tech_max  = minmax_scale(tech)
sent_scaled,  sent_min,  sent_max  = minmax_scale(sent)
macro_scaled, macro_min, macro_max = minmax_scale(macro)

labels_scaled, price_min, price_max = minmax_scale(labels)

np.savez_compressed(
    "aligned_data.npz",
    tech=tech_scaled, sent=sent_scaled,
    macro=macro_scaled, price=labels_scaled)


class MultiCSVWindowDataset(Dataset):
    """
    Each sample = (x_stock, x_sent, x_macro, y)
    Shapes:
      x_* : [window, F]   (one ticker, consecutive dates)
      y   : [horizon, 4]  (open/close/high/low for next horizon days)
    """
    def __init__(self, npz_path: str, window: int = WINDOW_SIZE, horizon: int = HORIZON_SIZE):
        data = np.load(npz_path)
        self.stock  = data['tech']    # [D, N, F1]
        self.sent   = data['sent']    # [D, N, F2]
        self.macro  = data['macro']   # [D, N, F3]
        self.labels = data['price']   # [D, N, 4]

        self.T = window
        self.H = horizon
        self.D = self.stock.shape[0] - window - horizon
        self.N = self.stock.shape[1]

    def __len__(self):
        return self.D * self.N            # all (ticker, start‑time) pairs

    def __getitem__(self, idx):
        t0 = idx % self.D                 # time start
        k  = idx // self.D                # ticker index

        x_stock = self.stock [t0:t0+self.T, k, :]
        x_sent  = self.sent  [t0:t0+self.T, k, :]
        x_macro = self.macro[t0:t0+self.T, k, :]
        y = self.labels[t0+self.T:t0+self.T+self.H, k, :]   # horizon×4

        return (torch.tensor(x_stock),
                torch.tensor(x_sent),
                torch.tensor(x_macro),
                torch.tensor(y))


#  usage example
dataset = MultiCSVWindowDataset("aligned_data.npz", window=WINDOW_SIZE, horizon=HORIZON_SIZE)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

for batch in loader:
    x_stock, x_sent, x_macro, y = batch   # ready for training step
    # print(batch)
    break



## Multiplex Attention Block

In [ ]:

class MultiplexAttention(nn.Module):
    """Applies self‑attention independently to three modality streams and concatenates the results."""

    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.attn_stock = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.attn_sent  = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.attn_macro = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        # projection after concat of 3*dim back to dim
        self.out_proj   = nn.Linear(3*dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x_stock, x_sent, x_macro, key_padding_mask=None, attn_mask=None):
        # each input: [B, T, D]
        a_stock, _ = self.attn_stock(x_stock, x_stock, x_stock,
                                     key_padding_mask=key_padding_mask,
                                     attn_mask=attn_mask)
        a_sent,  _ = self.attn_sent(x_sent, x_sent, x_sent,
                                    key_padding_mask=key_padding_mask,
                                    attn_mask=attn_mask)
        a_macro, _ = self.attn_macro(x_macro, x_macro, x_macro,
                                     key_padding_mask=key_padding_mask,
                                     attn_mask=attn_mask)
        # concat on last dim
        cat = torch.cat([a_stock, a_sent, a_macro], dim=-1)
        out = self.out_proj(cat)
        return self.norm(out)


## Transformer Forecasting Model

In [ ]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class MultiplexTransformerForecaster(nn.Module):
    def __init__(self,
                 dim: int = 128,
                 num_heads: int = 4,
                 num_layers: int = 4,
                 horizon: int = HORIZON_SIZE,
                 dropout: float = 0.1,
                 tech_dim:  int = TECH_DIM,
                 sent_dim:  int = SENT_DIM,
                 macro_dim: int = MACRO_DIM,
                 window:    int = WINDOW_SIZE):
        super().__init__()
        self.horizon = horizon
        self.window  = window

        # modality‑specific projections
        self.embed_stock = nn.Linear(tech_dim,  dim)
        self.embed_sent  = nn.Linear(sent_dim,  dim)
        self.embed_macro = nn.Linear(macro_dim, dim)

        self.pos_enc = PositionalEncoding(dim)

        blocks = []
        for _ in range(num_layers):
            blocks.append(MultiplexAttention(dim, num_heads))
            blocks.append(nn.Sequential(
                nn.LayerNorm(dim),
                nn.Linear(dim, dim*4), nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(dim*4, dim)
            ))
        self.layers = nn.ModuleList(blocks)

        # prediction head: produce horizon × 4 prices, not just horizon
        self.pred_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Flatten(start_dim=1),                      # [B, window*dim]
            nn.Linear(dim * window, horizon * 4)
        )

    def forward(self, x_stock, x_sent, x_macro):          # [B, T, F*]
        s = self.embed_stock(x_stock)
        t = self.embed_sent(x_sent)
        m = self.embed_macro(x_macro)

        s, t, m = map(self.pos_enc, (s, t, m))

        for blk in self.layers:
            if isinstance(blk, MultiplexAttention):
                residual = s
                s = t = m = blk(s, t, m)
                s = s + residual
            else:                         # feed‑forward
                s = s + blk(s)
                t = m = s

        out = self.pred_head(s)                           # [B, horizon*4]
        return out.view(-1, self.horizon, 4)              # [B, H, 4]


### Inject LoRA Adapters

In [ ]:

# Uncomment to add LoRA adapters if `peft` is installed in environment
# from peft import get_peft_model, LoraConfig, TaskType
# lora_cfg = LoraConfig(
#     task_type=TaskType.FEATURE_EXTRACTION,
#     target_modules=['q_proj', 'k_proj', 'v_proj'],
#     r=8, lora_alpha=32, lora_dropout=0.05)
# model = MultiplexTransformerForecaster().to(device)
# model = get_peft_model(model, lora_cfg)


## Training Loop

In [ ]:

def train_epoch(model, loader, optimizer, scaler, loss_fn, grad_clip=1.0):
    model.train()
    total_loss = 0
    for x_stock, x_sent, x_macro, y in loader:
        x_stock = x_stock.to(device)
        x_sent  = x_sent.to(device)
        x_macro = x_macro.to(device)
        y = y.to(device)
        with torch.cuda.amp.autocast():
            y_hat = model(x_stock, x_sent, x_macro)
            loss = loss_fn(y_hat, y)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * y.size(0)
    return total_loss / len(loader.dataset)


In [ ]:

# Example skeleton training script (adjust paths & hyperparams)
dataset = MultiCSVWindowDataset('aligned_data.npz',
                                window=WINDOW_SIZE,
                                horizon=HORIZON_SIZE)
loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)
model = MultiplexTransformerForecaster(
            tech_dim=TECH_DIM,
            sent_dim=SENT_DIM,
            macro_dim=MACRO_DIM,
            window=WINDOW_SIZE,
            horizon=HORIZON_SIZE).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
loss_fn = nn.MSELoss()
scaler = torch.cuda.amp.GradScaler()

# # for debugging purposes
# batch = next(iter(loader))
# for name, tensor in zip(
#         ["stock","sent","macro","label"], batch):
#     print(name, "nan =", torch.isnan(tensor).any().item(),
#                 "inf =", torch.isinf(tensor).any().item(),
#                 "max =", tensor.max().item())


for epoch in range(20):
    train_loss = train_epoch(model, loader, optimizer, scaler, loss_fn)
    print(f'Epoch {epoch+1}: train MSE = {train_loss:.4f}')


## Save Trained Model

In [ ]:

# torch.save(model.state_dict(), 'multiplex_transformer.pth')


### Next Steps and Integration
- Add an evaluation loop on the validation/test split.
- Export the frozen Transformer to feed the RL environment.
- Experiment with different window/horizon sizes, LoRA ranks, and learning rates.